# Prompt Engineering Best Practices: Text Transforming & Translation

[Article](https://todatabeyond.com/blog/prompt-engineering-best-practices-text-transforming-and-translation). [Series contents](./README.md).

Install `pip install openai==2.43.0 python-dotenv==1.2.2 bleach==6.4.0 beautifulsoup4==4.14.3 redlines==0.6.2 Markdown==3.10`. Set OPENAI_API_KEY in the environment or an uncommitted local .env. MODEL defaults to gpt-5.4-mini. Calls incur charges; never put a key in cells. Original code and output remain in the nonexecuting appendix. Examples keep the complete source input texts and loops; outputs can change between invocations.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion(prompt, model=MODEL):
    result = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        reasoning_effort="none", max_completion_tokens=2048)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def parse_json_object(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    raise ValueError("No JSON object found")
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
prompt = f"""
Translate the following English text to Arabic:  ```Hi, I would like to order a new book```
"""
response = get_completion(prompt)
print(response)
arabic_translation = response

In [ ]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)
language_response = response

In [ ]:
prompt = f"""
Translate the following  text to French and Spanish
and Arabic:  ```I want to order a book and get it delivered to my home and make sure that the book is packaged in a good way so it will arrive in a good form```
"""
response = get_completion(prompt)
print(response)
multilingual_response = response

In [ ]:
prompt = f"""
Translate the following text to Arabic in both the  formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)
formal_response = response

In [ ]:
support_results = []
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 


for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English      and Arabic: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")
    support_results.append({"issue": issue, "language": lang, "translation": response})

In [ ]:
prompt = f"""
Transform the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)
business_response = response

In [ ]:
data_json = { "restaurant employees" :[ 
    {"name":"Shyam", "email":"shyam@example.invalid"},
    {"name":"Bob", "email":"bob@example.invalid"},
    {"name":"Jai", "email":"jai@example.invalid"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML  table with column headers and title: {json.dumps(data_json)}
"""
prompt += "\nReturn only an HTML fragment with a heading and a table. Preserve every supplied name and address exactly; do not add CSS, scripts or links."
response = get_completion(prompt)
print(response)
employee_html = response

In [ ]:
import bleach
from bs4 import BeautifulSoup
from IPython.display import display, HTML
table_tags = ["h2", "h3", "table", "caption", "thead", "tbody", "tr", "th", "td"]
safe_employee_html = bleach.clean(employee_html, tags=table_tags, attributes={}, strip=True)
table_document = BeautifulSoup(safe_employee_html, "html.parser")
employee_rows = [(cells[0].get_text(strip=True), cells[1].get_text(strip=True))
                 for row in table_document.select("tr") if len(cells := row.select("td")) == 2]
print(safe_employee_html)
display(HTML(safe_employee_html))

In [ ]:
grammar_results = []
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    any errors, just say "No errors found". Return only the corrected sentence without surrounding quotation marks. Keep necessary punctuation inside the sentence, including apostrophes and question marks:
    ```{t}```"""
    response = get_completion(prompt)
    grammar_results.append(response)
    print(response)

In [ ]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking  mine from my room.  Yes, adults also like pandas too.  She takes  it everywhere with her, and it's super soft and cute.  One of the  ears is a bit lower than the other, and I don't think that was  designed to be asymmetrical. It's a bit small for what I paid for it  though. I think there might be other options that are bigger for  the same price.  It arrived a day earlier than expected, so I got  to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)
proofread_response = response

In [ ]:
from redlines import Redlines
from bleach.css_sanitizer import CSSSanitizer
diff = Redlines(text, proofread_response)
safe_diff = bleach.clean(diff.output_markdown, tags=["span"],
    attributes={"span": ["style"]}, strip=True,
    css_sanitizer=CSSSanitizer(allowed_css_properties=["color", "font-weight", "text-decoration"]))
print(safe_diff)
display(HTML(safe_diff))

In [ ]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
print(response)
rewritten_review = response
import markdown
review_tags = ["h1", "h2", "h3", "p", "strong", "em", "ul", "ol", "li", "blockquote", "br"]
safe_review_html = bleach.clean(markdown.markdown(rewritten_review), tags=review_tags, attributes={}, strip=True)
rewritten_document = BeautifulSoup(safe_review_html, "html.parser")
display(HTML(safe_review_html))

In [ ]:
assert re.search(r"[\u0600-\u06ff]", arabic_translation)
assert "french" in language_response.lower()
assert all(label in multilingual_response.lower() for label in ["french", "spanish", "arabic"])
assert re.search(r"[\u0600-\u06ff]", formal_response)
expected_languages = ["french", "spanish", "italian", "polish", "chinese"]
expected_terms = ["performance", "pixel", "mouse", "key", "screen"]
assert len(support_results) == 5
for result, language, term in zip(support_results, expected_languages, expected_terms, strict=True):
    assert language in result["language"].lower() or (language == "chinese" and "mandarin" in result["language"].lower())
    assert term in result["translation"].lower()
    assert re.search(r"[\u0600-\u06ff]", result["translation"])
assert "joe" in business_response.lower() and "lamp" in business_response.lower()
assert len(employee_rows) == 3
expected_rows = [(r["name"], r["email"]) for r in data_json["restaurant employees"]]
assert employee_rows == expected_rows
assert not table_document.select("script,iframe,a,img")
assert not any(t.attrs for t in table_document.find_all())
assert len(grammar_results) == 7
normal = [t.replace("’", "'").lower() for t in grammar_results]
assert "has a ball" in normal[0]
assert normal[1].strip().rstrip(".") == "no errors found"
assert ("it's going" in normal[2] or "it is going" in normal[2]) and "its oil" in normal[2]
assert "there goes" in normal[3] and "their suitcases" in normal[3]
assert "they're going" in normal[3] or "they are going" in normal[3]
assert ("you're going" in normal[4] or "you are going" in normal[4]) and "your notebook" in normal[4]
assert "affects" in normal[5] and "butterfly effect" in normal[5]
assert all(s in normal[6] for s in ["check", "spelling", "ability"])
assert all(s in proofread_response.lower() for s in ["daughter", "birthday", "ear", "small"])
assert "span" in safe_diff and "line-through" in safe_diff
diff_document = BeautifulSoup(safe_diff, "html.parser")
assert not diff_document.select("script,iframe,a,img")
assert all(set(t.attrs) <= {"style"} for t in diff_document.find_all())
assert "panda" in rewritten_review.lower()
assert not rewritten_document.select("script,iframe,a,img")
assert not any(t.attrs for t in rewritten_document.find_all())
probe = bleach.clean('<script>alert(1)</script><img src=x onerror=bad()><table onclick="bad()"><tr><td>safe</td></tr></table>', tags=table_tags, attributes={}, strip=True)
assert not BeautifulSoup(probe, "html.parser").select("script,img")
print("PASS: five source languages and Arabic-script output, three exact reserved-address rows, all seven grammar cases, proofread meaning anchors, real Redlines diff and sanitized HTML/Markdown structure. These checks do not certify universal translation or APA-style accuracy.")

## Original archive appendix — not executed

Historical SDK fragments, examples and transcripts below are retained verbatim, not used as fresh test results.

### Original pre block 1

```text
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')


def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

### Original pre block 2

```text
prompt = f"""
Translate the following English text to Arabic: \ 
```Hi, I would like to order a new book```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 3

```text
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 4

```text
prompt = f"""
Translate the following  text to French and Spanish
and Arabic: \
```I want to order a book and get it delivered to my home and make sure that the book is packaged in a good way so it will arrive in a good form```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 5

```text
prompt = f"""
Translate the following text to Arabic in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 6

```text
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 


for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Arabic: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")
```

### Original pre block 7

```text
prompt = f"""
Transform the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 8

```text
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 9

```text
<!DOCTYPE html>
<html>
<head>
<style>
table {
  font-family: arial, sans-serif;
  border-collapse: collapse;
  width: 100%;
}

td, th {
  border: 1px solid #dddddd;
  text-align: left;
  padding: 8px;
}
tr:nth-child(even) {
  background-color: #dddddd;
}
</style>
</head>
<body>
<h2>Restaurant Employees</h2>
<table>
  <tr>
    <th>Name</th>
    <th>Email</th>
  </tr>
  <tr>
    <td>Shyam</td>
    <td>shyamjaiswal@gmail.com</td>
  </tr>
  <tr>
    <td>Bob</td>
    <td>bob32@gmail.com</td>
  </tr>
  <tr>
    <td>Jai</td>
    <td>jai87@gmail.com</td>
  </tr>
</table>
</body>
</html>
```

### Original pre block 10

```text
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))
```

### Original pre block 11

```text
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)
```

### Original pre block 12

```text
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)
```

### Original pre block 13

```text
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))
```

### Original pre block 14

```text
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))
```

### Original transcript 1

```text
مرحبًا ، أود أن أطلب كتابًا جديدًا
```

### Original transcript 2

```text
This language is French.
```

### Original transcript 3

```text
French: Je veux commander un livre et le faire livrer chez moi en m’assurant que le livre est bien emballé afin qu’il arrive en bon état.
```

### Original transcript 4

```text
Spanish: Quiero pedir un libro y que me lo entreguen en mi casa, asegurándome de que el libro esté bien embalado para que llegue en buen estado.
```

### Original transcript 5

```text
Arabic: أرغب في طلب كتاب وتوصيله إلى منزلي والتأكد من أن الكتاب معبأ بشكل جيد حتى يصل بحالة جيدة.
```

### Original transcript 6

```text
Formal: هل ترغب في طلب وسادة؟
Informal: تبغى تطلب وسادة؟
```

### Original transcript 7

```text
Original message (The language is French.): La performance du système est plus lente que d’habitude.
The performance of the system is slower than usual.
```

### Original transcript 8

```text
أداء النظام أبطأ من المعتاد.
```

### Original transcript 9

```text
Original message (The language is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English translation: “My monitor has pixels that do not light up.”
```

### Original transcript 10

```text
Arabic translation: “شاشتي تحتوي على بكسلات لا تضيء.”
```

### Original transcript 11

```text
Original message (The language is Italian.): Il mio mouse non funziona
English: “My mouse is not working.”
Arabic: “فأرتي لا تعمل.”
```

### Original transcript 12

```text
Original message (The language is Polish.): Mój klawisz Ctrl jest zepsuty
English translation: “My Ctrl key is broken”
```

### Original transcript 13

```text
Arabic translation: “مفتاح Ctrl الخاص بي معطل”
```

### Original transcript 14

```text
Original message (The language is Chinese.): 我的屏幕在闪烁
English translation: “My screen is flickering.”
```

### Original transcript 15

```text
Arabic translation: “شاشتي تومض.”
```

### Original transcript 16

```text
Dear Sir/Madam,
```

### Original transcript 17

```text
I hope this letter finds you well. My name is Joe, and I am writing to bring your attention to a specification document regarding a standing lamp.
```

### Original transcript 18

```text
I kindly request that you take a moment to review the attached spec, as it contains important details about the standing lamp in question.
```

### Original transcript 19

```text
Thank you for your time and consideration. I look forward to hearing from you soon.
```

### Original transcript 20

```text
Sincerely,
Joe
```

### Original transcript 21

```text
The girl with the black and white puppies has a ball.
No errors found.
It’s going to be a long day. Does the car need its oil changed?
There goes my freedom. They’re going to bring their suitcases.
You’re going to need your notebook.
That medicine affects my ability to sleep. Have you heard of the butterfly effect?
This phrase is to check chatGPT for spelling ability.
```

### Original transcript 22

```text
Got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it’s super soft and cute. However, one of the ears is a bit lower than the other, and I don’t think that was designed to be asymmetrical. Additionally, it’s a bit small for what I paid for it. I believe there might be other options that are bigger for the same price. On the positive side, it arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.
```

### Original transcript 23

```text
Review of a Panda Plush Toy: A Perfect Gift for All Ages
```

### Original transcript 24

```text
I purchased this delightful panda plush toy as a birthday gift for my daughter, who has a penchant for “borrowing” my belongings from my room. It is worth noting that pandas are not only adored by children but also by adults, and this toy certainly did not disappoint.
```

### Original transcript 25

```text
The first thing that struck me about this plush toy is its exceptional softness and undeniable cuteness. Its irresistibly cuddly texture makes it the perfect companion for my daughter, who takes it everywhere she goes. However, I did notice a slight asymmetry in the ears, which I believe was not intentionally designed that way. While this does not detract from its overall appeal, it is worth mentioning for the sake of accuracy.
```

### Original transcript 26

```text
In terms of size, I must admit that I expected a slightly larger toy considering its price. It is worth exploring other options that offer a more substantial size for the same price point. However, this minor setback did not overshadow the joy it brought to my daughter.
```

### Original transcript 27

```text
Furthermore, I was pleasantly surprised when the package arrived a day earlier than expected. This unexpected early delivery allowed me to indulge in some quality playtime with the panda plush toy before presenting it to my daughter. The anticipation and excitement it generated were truly priceless.
```

### Original transcript 28

```text
In conclusion, this panda plush toy is a delightful gift that transcends age boundaries. Its exceptional softness, undeniable cuteness, and portability make it a perfect companion for anyone, young or old. While there may be other options available in terms of size, the overall quality and joy it brings are undeniable. I highly recommend this panda plush toy as a gift that will undoubtedly bring smiles and warmth to the heart of its recipient.
```